# Project 1 – To Do List



## Problem Definition

1. Write a concise problem definition for the project. Put it in a text field at the top of your Jupyter notebook.



In [ ]:
# @title we want to predict whether or not a future customer will make a transaction based on their transaction data.



## Data Collection

2. Load Pandas, Numpy, and Matplotlib.

1. Load data Train.csv from AWS S3.



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


## rwc - Why split the url in two?

In [ ]:
base_data = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.1-Transactions/Data/Transaction.train.big.csv"
transactions = base_data + "train.csv"

## Data Cleaning

4. Examine the data using tools we have used in class.

1. If there are data cleaning issues, develop recommendations for how to deal with them.



In [ ]:
transactions = pd.read_csv(base_data)
transactions

In [ ]:
# info about data
transactions.info()

In [ ]:
#flipped the graph for no reason this doesnt help since we had a lot of columns and rows
transactions.transpose()

In [ ]:
transactions.head(5)

In [ ]:
transactions.tail(5)

In [ ]:
# their are rows and columns that are filled with more than 100 NaN's. lets find out how to get rid of them. our "target" seems to only ne filled by 1.0, 0.0, and Nan.

In [ ]:
transactions["target"].value_counts( dropna= False)
#tells us how many people did not make a transaction (161960) and how many people did complete a transaction(18040)

In [ ]:
transactions["target"].value_counts( normalize=True, dropna= False).mul(100)
#this gives us the percentage of how many people have and have not made a transactions.
# 15.42 % of people deid not purchase
# 1.72 % made a transaction
# 82.86% of the data is NaN

In [ ]:
# this gives me the total NaN's from each columns
transactions.isna().sum().sort_values(ascending=False).mul(1)

In [ ]:
# relative frequency is done to better see the scale of how many nulls we have
transactions.isna().sum().sort_values(ascending=False).mul(100).div(len(transactions))

In [ ]:
#tells us what the data is, strings or integers or in this case floats
transactions.dtypes.sort_values()

In [ ]:
#tells us how many columns and rows are in the data ( Its demensions)
transactions.shape

In [ ]:
# tells us how many elemts are in the data ( rows multiplied by the row ( Rows X columns))
transactions.size

In [ ]:
#gives us our unique values. IF THE NUMBERS ARE VERY HIGH WE MUST TAKE THEM OUT OF THE DATA.
#pull out "Unnamed: 0", "ID_code", i would even consider dropping var_52 86 and 94.
nunique = transactions.nunique().sort_values( ascending = False )
nunique

In [ ]:
# this shows us again what rows / columns have purely nulls. it tells us directly what needs to be dropped
drop_data = nunique[ nunique == transactions.shape[0] ].index
drop_data

In [ ]:
2 # two characters are unique they return 1050000 nulls in there given row/column

In [ ]:
# now lets beggin dropping the NaN's
# lets beggin with cleaning our target, then dropping the,"unnamed:0" " and "ID_code" then I will move on to cleaning the rest of the data that has some nulls

In [ ]:
# lets make a copy of the original data before we start dropping things. a back up or something to compare my progress  with
transac = transactions.copy()
transac

In [ ]:
transactions = transactions.drop( columns = [ "Unnamed: 0", "ID_code",])

In [ ]:
[transac["target"].info]

In [ ]:
transactions = transactions[transactions["target"].notna()]
print(transactions.shape)

In [ ]:
transactions["target"].value_counts( dropna=False)

In [ ]:
transac["target"].value_counts( dropna= False)

In [ ]:
transactions.duplicated().sum()

In [ ]:
#my rows from unnamed - var 100 were not touched and we must change that
print(transactions.columns.to_list())
print(transactions.shape)

In [ ]:
print(transactions.columns[transactions.isnull().all()].to_list())

In [ ]:
# these are the rows that are 100% full of nulls. i know this because the demension of transactions are 180000(columns) X 104 (rows)
print(transactions.isnull().sum()[transactions.isnull().sum()>0])

In [ ]:
# we have dropped 51 nulled rows
null_columns = transactions.columns[transactions.isnull().all()].to_list()
transactions = transactions.drop(columns = null_columns)
print( f" drop {len(null_columns)} complete null columns")
print(transactions.shape)

In [ ]:
# we have 0 duplicates meaning no more nulls
print(transactions.isnull().sum().sum())
print(transactions.duplicated().sum())

In [ ]:
missing_counts = transactions.isnull().sum()
print(missing_counts[missing_counts > 0])


In [ ]:
transactions

In [ ]:
# no more True output that there is NaN's values in the data
transactions.isna().sum().sort_values(ascending=False).mul(100).div(len(transactions))

## Exploratory Data Analysis

6. Produce some visual analysis of the data – like plots showing the distributions of all variables. Recall that Gaussian Naive Bayes assumes the predictors are normally distributed. Note: you might have to do multiple plots in groups.

1. NOTE: the ‘target’ column indicates a successful transaction (‘1’) or a no-transaction (‘0’). Verify these are the only values in that column.

1. Check the correlation values between all **predictor columns** to ensure there are no substantial correlations between predictors. This is important to support the decision to classify the ‘target’ using Naïve Bayes.

1. Create two data frames: one with all successful transactions, one with all unsuccessful transactions. **Make sure they are copies and not slices**.





In [ ]:
transactions_graph = transactions
transactions_graph.hist(figsize=(20, 40), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
corr = transactions.corr()
corr

In [ ]:
plt.figure(figsize=(10,10))
sns.heatmap(corr, cmap='viridis',annot = True, vmin = -1 ,vmax = 1);
# they are all pretty close to 0

## Data Processing

10. Create two data frames: one with all the predictor columns (everything except for Unnamed: 0, ID_code and target) and one with just the target. Make sure they are copies and not slices.





In [ ]:
transactions_drop = transactions.drop(columns=['target']).copy()
transactions_drop

In [ ]:
transactions_tar = transactions[['target']].copy()
transactions_tar

## Data Processing

10. Create two data frames: one with all the predictor columns (everything except for Unnamed: 0, ID_code and target) and one with just the target. Make sure they are copies and not slices.

1. Define a Gaussian Naïve Bayes model using Sklearn.

1. Divide the two data frames you created in step #10 into training and testing subsets.

1. Train the model using the training subset of the dataset.

1. Test the model using the testing subset of the dataset. Calculate and report the accuracy.

1. Perform a cross-validation loop to calculate the accuracy of your model. Report that accuracy. How does it compare to the accuracy you calculated in #14?

1. Plot a histogram of the accuracy scores you generated in your cross-validation loop. What do you notice about the distribution of accuracy scores?

1.  Present the confusion matrix and the results of your Classification Report (sklearn.metrics.classification_report). What do you notice?

1. The training data is very skewed towards non-successful transactions (about 90% of the training data has ‘target’==0). Remove enough non-successful transaction rows so that your remaining training data is 50%/50% split between successful and non-successful transactions. Hint: you can use the data frames you created in step #9.

1. Repeat the cross-validation process on this data set. Report what your cross-validation accuracy is in this 50/50 case.



In [ ]:
#from sklearn import datasets
#from sklearn import metrics
#from sklearn import preprocessing
#import sklearn.model_selection

In [ ]:

from sklearn.naive_bayes import GaussianNB
model = GaussianNB()
print(model)

In [ ]:
from sklearn.model_selection import train_test_split

##



12. Divide the two data frames you created in step #10 into training and testing subsets.


In [ ]:
x = transactions.drop(columns = ["target"])
y = transactions['target']

In [ ]:
# testing size is 30%
X_train, X_test, y_train, y_test = train_test_split(x,y, test_size = 0.3, random_state = 42)

In [ ]:
# will be using 54,000 samples to train and 126000 to train for the var_1 - var100
print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

In [ ]:
model.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import accuracy_score
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

In [ ]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(model, x, y, cv=5)
print(f"Cross-validation scores: {scores}")
print(f"Mean accuracy: {scores.mean()}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(scores, bins=5, edgecolor='black')
plt.title('Cross-Validation Accuracy Scores')
plt.xlabel('Accuracy')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# f1-scores should be 50/50
from sklearn.metrics import confusion_matrix, classification_report
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
transactions_balanced = pd.concat([
    transactions,
    transactions.sample(n=len(transactions), random_state=42)
])
print(transactions['target'].value_counts())
print(transactions.shape)

In [ ]:
# balance is still the same
x = transactions.drop(columns = ["target"])
y = transactions['target']
scores_bal = cross_val_score(model, x, y , cv=5)
print(f"Balanced CV scores: {scores_bal}")
print(f"Balanced mean accuracy: {scores_bal.mean()}")

In [ ]:
#w = transactions_tar
#z = transactions['target']

In [ ]:
# testing size is 30%. random_state keeps the results the same
#w_train, w_test, z_train, z_test = train_test_split(w,z, test_size = 0.3, random_state = 42)

In [ ]:
#print("Training features shape:", w_train.shape)
#print("Testing features shape:", w_test.shape)

In [ ]:
#from sklearn.linear_model import LinearRegression
#clf = LinearRegression()

In [ ]:
#clf.fit(x_train,y_train)

In [ ]:
#y_test

In [ ]:
#says accuracy is 100% for columns
#clf.score(x_test,y_test)

In [ ]:
#clf.fit(w_train,w_train)

In [ ]:
#clf.predict(w_test)

In [ ]:
#100% accuracy
#clf.score(w_test,z_test)

In [ ]:
#lets tes it now

In [ ]:
from sklearn.model_selection import KFold
kf = KFold(n_splits=3)
kf

In [ ]:
for train_index, test_index in kf.split([1,2,3,4,5,6]):
  print(train_index, test_index)

In [ ]:
##################################################################################################

In [ ]:
transactions_tar
target_list = transactions_tar['target'].tolist()
target_list

In [ ]:
#transactions_drop
#drop_list = transactions_drop['var_0'].tolist()
#drop_list

## Data Visualization


20. Compare the results of your cross-validation with the whole training data and the reduced 50/50 training data

1. Present the confusion matrix and the results of your Classification Report (sklearn.metrics.classification_report)




In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(['Full Data (90/10)', '50/50 Balanced'],
        [scores.mean(), scores_bal.mean()],
        color=['steelblue', 'darkorange'],
        edgecolor='black')
plt.title('Cross-Validation Accuracy: Full vs Balanced')
plt.ylabel('Mean Accuracy')
plt.ylim(0.5, 1.0)
plt.show()

## rwc - not sure what this does, but it doesn't work.  

Looks like the X_bal variable isn't defined.


In [ ]:
X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42)
model.fit(X_train_bal, y_train_bal)
y_pred_bal = model.predict(X_test_bal)
print(confusion_matrix(y_test_bal, y_pred_bal))
print(classification_report(y_test_bal, y_pred_bal))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
# Imbalanced model confusion matrix
cm = confusion_matrix(y_test, y_pred)
# Balanced model confusion matrix
cm_bal = confusion_matrix(y_test_bal, y_pred_bal)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='GnBu', ax=axes[0])
axes[0].set_title('Imbalanced Model')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')
sns.heatmap(cm_bal, annot=True, fmt='d', cmap='GnBu', ax=axes[1])
axes[1].set_title('Balanced Model')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
plt.tight_layout()
plt.show()

## Communicate the Results

22. Communicate the results of your analysis.



## Submit Final Project

23. Upload your finished Jupyter notebook to your Project 1 student folder.
